# Object Detection with PyTorch

Detect and localize objects in images using pre-trained models.

## Learning Objectives

- Understand object detection concepts
- Use pre-trained Faster R-CNN for detection
- Visualize bounding boxes and predictions
- Learn about detection metrics (IoU, mAP)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

import torchvision
from torchvision import datasets
from torchvision.transforms import v2
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
)
from torchvision.utils import draw_bounding_boxes

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"torchvision version: {torchvision.__version__}")

## 1. Object Detection Concepts

Object detection combines:
- **Classification**: What objects are in the image?
- **Localization**: Where are the objects (bounding boxes)?

### Output Format
```
detection = {
    'boxes': [[x1, y1, x2, y2], ...],  # Top-left and bottom-right corners
    'labels': [class_id, ...],          # Class indices
    'scores': [confidence, ...],        # Detection confidence
}
```

In [ ]:
# Detection architectures overview
architectures = """
=== Object Detection Architectures ===

┌─────────────────┬────────────────────────────────────────────────┐
│ Type            │ Description                                    │
├─────────────────┼────────────────────────────────────────────────┤
│ Two-Stage       │ First generate proposals, then classify       │
│ (Faster R-CNN)  │ Higher accuracy, slower                        │
├─────────────────┼────────────────────────────────────────────────┤
│ One-Stage       │ Direct prediction of boxes and classes        │
│ (YOLO, SSD)     │ Faster, good for real-time                    │
├─────────────────┼────────────────────────────────────────────────┤
│ Anchor-Free     │ No predefined anchor boxes                    │
│ (FCOS, CenterNet│ Simpler, competitive accuracy                 │
└─────────────────┴────────────────────────────────────────────────┘

Available in torchvision:
  • Faster R-CNN (ResNet-50 FPN, MobileNet V3)
  • FCOS (ResNet-50 FPN)
  • RetinaNet (ResNet-50 FPN)
  • SSD (VGG-16)
  • SSDLite (MobileNet V3)
"""
print(architectures)

## 2. Load Pre-trained Model

We'll use Faster R-CNN with a ResNet-50 FPN backbone, trained on COCO.

In [ ]:
# Load Faster R-CNN model
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=weights)
model = model.to(device)
model.eval()

# Get COCO class labels
coco_labels = weights.meta['categories']
print(f"Model trained on {len(coco_labels)} COCO classes")
print(f"\nFirst 20 classes: {coco_labels[:20]}")

In [ ]:
# Get preprocessing transforms from weights
preprocess = weights.transforms()
print("Preprocessing transforms:")
print(preprocess)

## 3. Create Test Images

We'll create synthetic images for demonstration.

In [ ]:
# Load sample images from CIFAR-10 and resize
cifar = datasets.CIFAR10(root='./datasets', train=False, download=True)

# Create a composite image with multiple objects
def create_composite_image(size=(640, 480)):
    """Create a simple synthetic image with colored shapes."""
    w, h = size
    
    # Create background
    img = np.ones((h, w, 3), dtype=np.uint8) * 200  # Light gray
    
    # Add some colored rectangles (simulating objects)
    # Red rectangle
    img[100:200, 100:250] = [255, 100, 100]
    
    # Green rectangle
    img[250:400, 300:500] = [100, 255, 100]
    
    # Blue rectangle
    img[50:180, 400:550] = [100, 100, 255]
    
    return Image.fromarray(img)

# Get a real image from CIFAR and upscale it
sample_pil, label = cifar[42]  # Get an image
sample_upscaled = sample_pil.resize((640, 480), Image.Resampling.LANCZOS)

print(f"Sample image class: {cifar.classes[label]}")

# Show both images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(sample_upscaled)
axes[0].set_title(f'CIFAR-10 Sample ({cifar.classes[label]})')
axes[0].axis('off')

synthetic = create_composite_image()
axes[1].imshow(synthetic)
axes[1].set_title('Synthetic Image')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 4. Run Object Detection

In [ ]:
def detect_objects(model, image, preprocess, device, threshold=0.5):
    """
    Run object detection on an image.
    
    Args:
        model: Detection model
        image: PIL Image
        preprocess: Preprocessing transforms
        device: torch device
        threshold: Confidence threshold
    
    Returns:
        Dictionary with boxes, labels, scores
    """
    # Convert to tensor
    img_tensor = v2.functional.to_image(image)
    
    # Preprocess
    batch = preprocess(img_tensor).unsqueeze(0).to(device)
    
    # Inference
    with torch.no_grad():
        predictions = model(batch)
    
    # Extract predictions above threshold
    pred = predictions[0]
    mask = pred['scores'] >= threshold
    
    return {
        'boxes': pred['boxes'][mask].cpu(),
        'labels': pred['labels'][mask].cpu(),
        'scores': pred['scores'][mask].cpu()
    }


# Run detection on CIFAR sample
results = detect_objects(model, sample_upscaled, preprocess, device, threshold=0.3)

print(f"Found {len(results['boxes'])} objects:")
for i in range(len(results['boxes'])):
    label_idx = results['labels'][i].item()
    score = results['scores'][i].item()
    box = results['boxes'][i].tolist()
    print(f"  {coco_labels[label_idx]}: {score:.1%} at {[int(x) for x in box]}")

In [ ]:
def visualize_detections(image, results, class_labels, figsize=(12, 8)):
    """
    Visualize object detections with bounding boxes.
    """
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(image)
    
    # Color map for different classes
    colors = plt.cm.Set3(np.linspace(0, 1, 12))
    
    for i in range(len(results['boxes'])):
        box = results['boxes'][i].numpy()
        label_idx = results['labels'][i].item()
        score = results['scores'][i].item()
        label = class_labels[label_idx]
        
        # Get box coordinates
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        
        # Draw rectangle
        color = colors[label_idx % len(colors)]
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Add label
        ax.text(
            x1, y1 - 5,
            f'{label}: {score:.1%}',
            fontsize=10, color='white',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.8)
        )
    
    ax.set_title(f'Detected {len(results["boxes"])} objects')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# Visualize
if len(results['boxes']) > 0:
    visualize_detections(sample_upscaled, results, coco_labels)
else:
    print("No objects detected above threshold")
    plt.imshow(sample_upscaled)
    plt.title("No detections")
    plt.axis('off')
    plt.show()

## 5. Using torchvision Drawing Utilities

In [ ]:
def draw_detections_torchvision(image, results, class_labels):
    """Use torchvision's built-in drawing utilities."""
    # Convert PIL to tensor (uint8)
    img_tensor = v2.functional.to_image(image)
    
    if len(results['boxes']) > 0:
        # Create labels with scores
        labels = [
            f"{class_labels[l.item()]}: {s:.1%}"
            for l, s in zip(results['labels'], results['scores'])
        ]
        
        # Draw boxes
        img_with_boxes = draw_bounding_boxes(
            img_tensor,
            boxes=results['boxes'],
            labels=labels,
            colors='red',
            width=3,
            font_size=14
        )
    else:
        img_with_boxes = img_tensor
    
    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(img_with_boxes.permute(1, 2, 0).numpy())
    plt.axis('off')
    plt.title('Detection with torchvision.utils.draw_bounding_boxes')
    plt.show()


# Draw with torchvision
draw_detections_torchvision(sample_upscaled, results, coco_labels)

## 6. Detection Metrics

Understanding how detection performance is measured.

In [ ]:
def compute_iou(box1, box2):
    """
    Compute Intersection over Union (IoU) between two boxes.
    
    Boxes in format [x1, y1, x2, y2]
    """
    # Intersection coordinates
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    # Intersection area
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    
    # Union area
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0


# Demonstrate IoU calculation
box_a = [100, 100, 200, 200]  # Ground truth
box_b = [150, 150, 250, 250]  # Prediction (partial overlap)
box_c = [100, 100, 200, 200]  # Perfect match
box_d = [300, 300, 400, 400]  # No overlap

print("=== IoU Examples ===")
print(f"Partial overlap: IoU = {compute_iou(box_a, box_b):.3f}")
print(f"Perfect match:   IoU = {compute_iou(box_a, box_c):.3f}")
print(f"No overlap:      IoU = {compute_iou(box_a, box_d):.3f}")

In [ ]:
# Visualize IoU
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

examples = [
    (box_a, box_b, 'Partial Overlap'),
    (box_a, box_c, 'Perfect Match'),
    (box_a, box_d, 'No Overlap')
]

for ax, (b1, b2, title) in zip(axes, examples):
    ax.set_xlim(0, 500)
    ax.set_ylim(500, 0)  # Flip y-axis to match image coordinates
    
    # Draw boxes
    rect1 = patches.Rectangle(
        (b1[0], b1[1]), b1[2]-b1[0], b1[3]-b1[1],
        linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.3,
        label='Ground Truth'
    )
    rect2 = patches.Rectangle(
        (b2[0], b2[1]), b2[2]-b2[0], b2[3]-b2[1],
        linewidth=2, edgecolor='red', facecolor='red', alpha=0.3,
        label='Prediction'
    )
    
    ax.add_patch(rect1)
    ax.add_patch(rect2)
    ax.set_title(f'{title}\nIoU = {compute_iou(b1, b2):.3f}')
    ax.legend()
    ax.set_aspect('equal')

plt.suptitle('Intersection over Union (IoU)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Detection metrics explanation
metrics_info = """
=== Object Detection Metrics ===

1. IoU (Intersection over Union)
   - Measures overlap between predicted and ground truth boxes
   - IoU > 0.5 typically considered a "match"

2. Precision and Recall
   - Precision = TP / (TP + FP)  (How many detections are correct?)
   - Recall = TP / (TP + FN)     (How many objects are found?)

3. Average Precision (AP)
   - Area under the Precision-Recall curve
   - AP@0.5: AP at IoU threshold 0.5
   - AP@0.75: AP at IoU threshold 0.75 (stricter)

4. Mean Average Precision (mAP)
   - Average of AP across all classes
   - COCO mAP: Average across IoU thresholds [0.5:0.95:0.05]

=== COCO Detection Benchmarks ===

┌─────────────────────────────────┬──────────┬─────────────┐
│ Model                           │ mAP      │ Speed (fps) │
├─────────────────────────────────┼──────────┼─────────────┤
│ Faster R-CNN ResNet-50 FPN V2   │ 46.7     │ ~7          │
│ Faster R-CNN MobileNet V3       │ 32.8     │ ~25         │
│ SSD300 VGG16                    │ 25.1     │ ~30         │
│ SSDLite320 MobileNet V3         │ 21.3     │ ~60         │
│ FCOS ResNet-50 FPN              │ 39.2     │ ~10         │
│ RetinaNet ResNet-50 FPN V2      │ 41.5     │ ~8          │
└─────────────────────────────────┴──────────┴─────────────┘
"""
print(metrics_info)

## 7. Faster Model for Real-time Detection

In [ ]:
# Load lighter model
mobile_weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
mobile_model = fasterrcnn_mobilenet_v3_large_fpn(weights=mobile_weights)
mobile_model = mobile_model.to(device)
mobile_model.eval()

mobile_preprocess = mobile_weights.transforms()

# Compare inference speed
import time

def benchmark_model(model, image, preprocess, device, num_runs=10):
    """Benchmark model inference speed."""
    img_tensor = preprocess(v2.functional.to_image(image)).unsqueeze(0).to(device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(3):
            _ = model(img_tensor)
    
    # Synchronize for accurate timing
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    start = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(img_tensor)
            if device.type == 'cuda':
                torch.cuda.synchronize()
    
    elapsed = time.time() - start
    return elapsed / num_runs * 1000  # ms per image


print("Benchmarking models...")
resnet_time = benchmark_model(model, sample_upscaled, preprocess, device)
mobile_time = benchmark_model(mobile_model, sample_upscaled, mobile_preprocess, device)

print(f"\n=== Inference Speed ===")
print(f"Faster R-CNN ResNet-50 FPN V2: {resnet_time:.1f} ms/image")
print(f"Faster R-CNN MobileNet V3:     {mobile_time:.1f} ms/image")
print(f"\nSpeedup: {resnet_time/mobile_time:.1f}x faster with MobileNet")

## 8. Non-Maximum Suppression (NMS)

NMS removes duplicate/overlapping detections.

In [ ]:
from torchvision.ops import nms, batched_nms

def explain_nms():
    """Demonstrate Non-Maximum Suppression."""
    # Simulate multiple overlapping detections
    boxes = torch.tensor([
        [100.0, 100.0, 200.0, 200.0],  # Detection 1 (high score)
        [105.0, 105.0, 210.0, 205.0],  # Detection 2 (overlapping, medium score)
        [95.0,  95.0,  195.0, 195.0],  # Detection 3 (overlapping, low score)
        [300.0, 300.0, 400.0, 400.0],  # Detection 4 (different object)
    ])
    
    scores = torch.tensor([0.95, 0.80, 0.60, 0.85])
    
    print("Before NMS:")
    for i, (box, score) in enumerate(zip(boxes, scores)):
        print(f"  Detection {i+1}: {box.tolist()} (score: {score:.2f})")
    
    # Apply NMS
    iou_threshold = 0.5
    keep_indices = nms(boxes, scores, iou_threshold)
    
    print(f"\nAfter NMS (IoU threshold={iou_threshold}):")
    for i in keep_indices:
        print(f"  Detection {i+1}: {boxes[i].tolist()} (score: {scores[i]:.2f})")
    
    return boxes, scores, keep_indices

boxes, scores, keep = explain_nms()

In [ ]:
# Visualize NMS
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['red', 'green', 'blue', 'orange']

# Before NMS
ax = axes[0]
ax.set_xlim(0, 500)
ax.set_ylim(500, 0)

for i, (box, score) in enumerate(zip(boxes.numpy(), scores.numpy())):
    rect = patches.Rectangle(
        (box[0], box[1]), box[2]-box[0], box[3]-box[1],
        linewidth=2, edgecolor=colors[i], facecolor=colors[i], alpha=0.3
    )
    ax.add_patch(rect)
    ax.text(box[0], box[1]-5, f'Det {i+1}: {score:.2f}', fontsize=10, color=colors[i])

ax.set_title('Before NMS (4 detections)')
ax.set_aspect('equal')

# After NMS
ax = axes[1]
ax.set_xlim(0, 500)
ax.set_ylim(500, 0)

for i in keep:
    box = boxes[i].numpy()
    score = scores[i].numpy()
    rect = patches.Rectangle(
        (box[0], box[1]), box[2]-box[0], box[3]-box[1],
        linewidth=3, edgecolor=colors[i], facecolor=colors[i], alpha=0.4
    )
    ax.add_patch(rect)
    ax.text(box[0], box[1]-5, f'Det {i+1}: {score:.2f}', fontsize=10, color=colors[i])

ax.set_title(f'After NMS ({len(keep)} detections)')
ax.set_aspect('equal')

plt.suptitle('Non-Maximum Suppression', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

### Key Concepts

| Concept | Description |
|---------|-------------|
| Bounding Box | [x1, y1, x2, y2] format for object location |
| IoU | Intersection/Union - overlap measure |
| NMS | Remove duplicate overlapping detections |
| mAP | Mean Average Precision - detection accuracy metric |

### Model Selection

| Use Case | Recommended Model |
|----------|------------------|
| Highest accuracy | Faster R-CNN ResNet-50 FPN V2 |
| Real-time (mobile) | SSDLite MobileNet V3 |
| Balance | Faster R-CNN MobileNet V3 |

In [ ]:
print("Notebook completed successfully!")